# Semantic Resume Shortlisting — Development Notebook
This notebook exposes the production workflow in a form that is easy to study and modify.

## 1. Model and score variables

In [ ]:
MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
SEMANTIC_WEIGHT = 0.55
SKILL_WEIGHT = 0.20
TFIDF_WEIGHT = 0.10
EXPERIENCE_WEIGHT = 0.10
EDUCATION_WEIGHT = 0.05
print('Total weight:', SEMANTIC_WEIGHT + SKILL_WEIGHT + TFIDF_WEIGHT + EXPERIENCE_WEIGHT + EDUCATION_WEIGHT)

## 2. Import the tested production functions

In [ ]:
from sentence_transformers import SentenceTransformer
from advanced_engine import (
    clean_text, chunk_text, extract_skills, extract_required_years,
    extract_candidate_years, education_level, rank_resumes_advanced
)

## 3. Example inputs
Replace these variables with the job and résumé text you want to test.

In [ ]:
JOB_DESCRIPTION = '''Data Analyst with 3+ years of experience in Python, SQL,
Power BI, ETL, data validation and stakeholder communication.
Bachelor's degree required.'''

RESUMES = [
    ('Candidate_A', '4 years using Python, SQL, pandas, PowerBI and data pipelines. Bachelor degree.'),
    ('Candidate_B', '5 years of graphic design using Photoshop and Illustrator.'),
]

## 4. Inspect extracted requirements

In [ ]:
print('JD skills:', sorted(extract_skills(JOB_DESCRIPTION)))
print('Required years:', extract_required_years(JOB_DESCRIPTION))
print('Required education:', education_level(JOB_DESCRIPTION))
print('JD chunks:', len(chunk_text(JOB_DESCRIPTION)))

## 5. Load the transformer
The first execution downloads the pretrained model. Later executions use the cached copy.

In [ ]:
model = SentenceTransformer(MODEL_NAME)

def encode_texts(texts):
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

## 6. Rank résumés

In [ ]:
results = rank_resumes_advanced(JOB_DESCRIPTION, RESUMES, encode_texts)
results

## 7. How the production score works
- Transformer embeddings understand similar meanings even when wording differs.
- Skill aliases connect terms such as `PowerBI` and `Power BI`, or `ETL` and `data pipelines`.
- TF-IDF retains exact terminology evidence.
- Experience and education are checked separately.
- Long documents are split into overlapping chunks to avoid transformer truncation.

Modify production weights inside `advanced_engine.py`, then validate the change here. For real accuracy measurement, prepare recruiter-reviewed expected rankings and compare the output against them.

## 8. OCR, authentication and database flow
- `document_parser.py` first attempts normal PDF text extraction. If fewer than 80 characters are found, EasyOCR reads rendered page images.
- `database.py` creates SQLite user and analysis tables. Passwords are stored as salted PBKDF2 hashes, never as plain text.
- `app.py` requires login and shows only the signed-in user's saved analyses.
- The database stores ranking results, not original résumé files or extracted résumé text.